In [1]:
import numpy as np

In [2]:
fingerprints = {
    "Hemingway wrote A Farewell to Arms": np.array([0.9, 0.8, 0.3, 0.0, 0.4]),
    "The Sun Also Rises, published 1926": np.array([0.6, 0.7, 0.0, 0.0, 0.9]),
    "World War I killed millions":        np.array([0.0, 0.0, 1.0, 0.0, 0.2]),
    "The cat sat on the mat":             np.array([0.0, 0.0, 0.0, 1.0, 0.0]),
}

In [3]:
question = "Who wrote A Farewell to Arms?"
q = np.array([0.85, 0.8, 0.2, 0.0, 0.3])
print("QUESTION:", question, "\nq(x) =", q, "\n")

QUESTION: Who wrote A Farewell to Arms? 
q(x) = [0.85 0.8  0.2  0.   0.3 ] 



In [4]:
print("DOT PRODUCT closeness:")
for text, d in fingerprints.items():
    print(f"  {np.dot(q, d):5.2f}   {text}")

DOT PRODUCT closeness:
   1.59   Hemingway wrote A Farewell to Arms
   1.34   The Sun Also Rises, published 1926
   0.26   World War I killed millions
   0.00   The cat sat on the mat


In [5]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("\nCOSINE closeness:")
for text, d in fingerprints.items():
    print(f"  {cosine(q, d):5.2f}   {text}")


COSINE closeness:
   1.00   Hemingway wrote A Farewell to Arms
   0.85   The Sun Also Rises, published 1926
   0.21   World War I killed millions
   0.00   The cat sat on the mat


In [2]:
library = [
    {"text": "Hemingway wrote A Farewell to Arms", "vec": np.array([0.9, 0.8, 0.3, 0.0, 0.4])},
    {"text": "The Sun Also Rises, published 1926", "vec": np.array([0.6, 0.7, 0.0, 0.0, 0.9])},
    {"text": "World War I killed millions",         "vec": np.array([0.0, 0.0, 1.0, 0.0, 0.2])},
    {"text": "The cat sat on the mat",              "vec": np.array([0.0, 0.0, 0.0, 1.0, 0.0])},
    {"text": "Fitzgerald wrote The Great Gatsby",   "vec": np.array([0.9, 0.8, 0.0, 0.0, 0.9])},
]

In [3]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [4]:
def search(q_vec, k=3):
    scored = [(cosine(q_vec, item["vec"]), item["text"]) for item in library]
    scored.sort(reverse=True)
    return scored[:k]

In [5]:
question = "Who wrote A Farewell to Arms?"
q = np.array([0.85, 0.8, 0.2, 0.0, 0.3])

In [6]:
print("QUESTION:", question, "\n")

QUESTION: Who wrote A Farewell to Arms? 



In [7]:
for rank, (score, text) in enumerate(search(q, k=3), 1):
    print(f"  #{rank}  score={score:.3f}   {text}")

  #1  score=0.995   Hemingway wrote A Farewell to Arms
  #2  score=0.912   Fitzgerald wrote The Great Gatsby
  #3  score=0.851   The Sun Also Rises, published 1926


In [8]:
raw_scores = np.array([0.995, 0.912, 0.851])
labels = ["Hemingway (z1)", "Fitzgerald (z2)", "Sun Also Rises (z3)"]

In [19]:
def softmax(scores, temperature=0.1):
    s = scores / temperature
    exps = np.exp(s)          # positive + exaggerate gaps
    return exps / exps.sum()

In [20]:
trust = softmax(raw_scores)
print("RAW -> TRUST (temperature=1):")
for lbl, r, t in zip(labels, raw_scores, trust):
    print(f"  {lbl:22s} raw={r:.3f}   trust={t:.3f}")
print(f"  sum = {trust.sum():.3f}\n")

RAW -> TRUST (temperature=1):
  Hemingway (z1)         raw=0.995   trust=0.598
  Fitzgerald (z2)        raw=0.912   trust=0.261
  Sun Also Rises (z3)    raw=0.851   trust=0.142
  sum = 1.000



In [12]:
print("Different TEMPERATURE:")
for temp in [0.1,0.3, 1.0, 5.0]:
    t = softmax(raw_scores, temperature=temp)
    print(f"  temp={temp:>4}: {t[0]:.3f} {t[1]:.3f} {t[2]:.3f}")

Different TEMPERATURE:
  temp= 0.1: 0.598 0.261 0.142
  temp= 0.3: 0.421 0.319 0.260
  temp= 1.0: 0.359 0.330 0.311
  temp= 5.0: 0.338 0.333 0.329


In [21]:
library = [
    {"text": "Hemingway wrote A Farewell to Arms", "vec": np.array([0.9,0.8,0.3,0.0,0.4])},
    {"text": "The Sun Also Rises, published 1926", "vec": np.array([0.6,0.7,0.0,0.0,0.9])},
    {"text": "World War I killed millions",         "vec": np.array([0.0,0.0,1.0,0.0,0.2])},
    {"text": "The cat sat on the mat",              "vec": np.array([0.0,0.0,0.0,1.0,0.0])},
    {"text": "Fitzgerald wrote The Great Gatsby",   "vec": np.array([0.9,0.8,0.0,0.0,0.9])},
]

In [22]:
def cosine(a,b): return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))
def softmax(s, temperature=1.0):
    e = np.exp(np.array(s)/temperature); return e/e.sum()

In [24]:
def retrieve(q_vec, k=3, temperature=0.1):
    scored = [(cosine(q_vec, it["vec"]), it) for it in library]
    scored.sort(key=lambda p: p[0], reverse=True)
    top = scored[:k]
    trust = softmax([s for s,_ in top], temperature)
    return [(t, it) for (t,(_,it)) in zip(trust, top)]

In [25]:
CANDIDATE_ANSWERS = ["Hemingway", "Fitzgerald", "Orwell", "nobody"]
def writer(question, passage_text, temperature=0.5):
    scores = [0.1 + (3.0 if a.lower() in passage_text.lower() else 0.0) for a in CANDIDATE_ANSWERS]
    return dict(zip(CANDIDATE_ANSWERS, softmax(scores, temperature)))

In [26]:
def rag_answer(question, q_vec, k=3):
    retrieved = retrieve(q_vec, k=k)
    blended = {a: 0.0 for a in CANDIDATE_ANSWERS}
    print(f"QUESTION: {question}\n" + "-"*60)
    for trust, item in retrieved:
        p_theta = writer(question, item["text"])
        for a in CANDIDATE_ANSWERS: blended[a] += trust * p_theta[a]
        best = max(p_theta, key=p_theta.get)
        print(f"  trust={trust:.3f} | {item['text'][:32]:32s} | ->'{best}'")
    print("-"*60)
    for a in sorted(blended, key=blended.get, reverse=True):
        print(f"    {a:11s} {blended[a]:.3f}")
    print(f"\n>>> RAG ANSWERS: '{max(blended, key=blended.get)}'")

In [27]:
rag_answer('Who wrote a Farewell to Arms?',np.array([0.85, 0.8, 0.2, 0.0, 0.3]), k=3)

QUESTION: Who wrote a Farewell to Arms?
------------------------------------------------------------
  trust=0.598 | Hemingway wrote A Farewell to Ar | ->'Hemingway'
  trust=0.260 | Fitzgerald wrote The Great Gatsb | ->'Fitzgerald'
  trust=0.142 | The Sun Also Rises, published 19 | ->'Hemingway'
------------------------------------------------------------
    Hemingway   0.629
    Fitzgerald  0.296
    Orwell      0.038
    nobody      0.038

>>> RAG ANSWERS: 'Hemingway'
